# 02 — Proper Motions

**Plots:** VPD (Gaia vs BP3M) · error ellipses · parallax distribution · PM vs position

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from pathlib import Path

from bp3m.pipeline.explore_utils import (
    load_gaia_catalog, load_bp3m_results, build_gaia_cov,
    bp3m_full_cov, gaia_pm_sigma, bp3m_pm_sigma, vpd_limits,
)
from bp3m.pipeline.catalog_utils import cov2_geom_sigma

DATA = Path(OUTPUT_DIR)

# Full Gaia catalog (used for standalone Gaia VPD only)
_gaia_files = sorted((DATA / FIELD_NAME / 'Gaia').glob('*_gaia.csv'))
gaia = load_gaia_catalog(_gaia_files[0])

# BP3M results — stars is a subset of gaia (only HST-detected stars)
# stars already contains all Gaia columns so use it for aligned computations
bp3m   = load_bp3m_results(DATA / FIELD_NAME / 'BP3M_results')
stars  = bp3m['stars']
C_full = bp3m_full_cov(bp3m)        # (N_stars, 5, 5) — aligned with stars
C_gaia_stars = build_gaia_cov(stars) # (N_stars, 5, 5) — aligned with stars

# 2-parameter Gaia solutions have no PM or parallax measurement — mask those
# entries so they don't appear as valid uncertainty estimates in plots
_gaia_2p = ~np.isfinite(stars['pmra'].values) | ~np.isfinite(stars['pmdec'].values)
C_gaia_stars = C_gaia_stars.copy()
C_gaia_stars[_gaia_2p, 2:, 2:] = np.nan   # PM and parallax blocks

has_gaia_pm      = np.isfinite(gaia['pmra'].values) & np.isfinite(gaia['pmdec'].values)
has_gaia_pm_s    = np.isfinite(stars['pmra'].values) & np.isfinite(stars['pmdec'].values)
print(f'Gaia catalog: {len(gaia)}   BP3M stars: {len(stars)}   with Gaia PMs: {has_gaia_pm_s.sum()}')

In [ ]:
# ── Side-by-side VPD: Gaia vs BP3M ───────────────────────────────────────────
pmra_g  = gaia['pmra'].values
pmdec_g = gaia['pmdec'].values
pmra_b  = stars['pmra_bp3m'].values
pmdec_b = stars['pmdec_bp3m'].values

xlim, ylim = vpd_limits(pmra_b, pmdec_b)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, pmra, pmdec, title, color in [
    (axes[0], pmra_g[has_gaia_pm], pmdec_g[has_gaia_pm],
     'Gaia DR3', 'steelblue'),
    (axes[1], pmra_b, pmdec_b, 'BP3M (this work)', 'darkorange'),
]:
    ax.scatter(pmra, pmdec, s=3, alpha=0.4, c=color, rasterized=True)
    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.3)
    ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.3)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)')
    ax.set_title(title)

axes[0].set_ylabel(r'$\mu_\delta$ (mas yr$^{-1}$)')
fig.suptitle(f'{FIELD_NAME} — Vector Point Diagram', fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_vpd.png', dpi=150)
plt.show()

In [ ]:
# ── VPD with error ellipses (bright subset) ───────────────────────────────────
# Use stars (aligned with C_full) — indices into stars, not the full gaia catalog
N_ELLIPSE = min(200, len(stars))
bright = np.argsort(stars['gmag'].values)[:N_ELLIPSE]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, pmra, pmdec, C_pm, title, fc in [
    (axes[0],
     stars['pmra'].values[bright], stars['pmdec'].values[bright],
     C_gaia_stars[bright, 2:4, 2:4],
     'Gaia DR3 (inflated)', 'steelblue'),
    (axes[1],
     pmra_b[bright], pmdec_b[bright],
     C_full[bright, 2:4, 2:4],
     'BP3M (this work)', 'darkorange'),
]:
    ax.scatter(pmra, pmdec, s=10, c=fc, alpha=0.7, zorder=3)
    for k in range(len(bright)):
        C2 = C_pm[k]
        if not np.all(np.isfinite(C2)):
            continue
        vals, vecs = np.linalg.eigh(C2)
        w, h = 2 * np.sqrt(np.maximum(vals, 0))
        if not (np.isfinite(w) and np.isfinite(h) and w > 0 and h > 0):
            continue
        angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
        ell = Ellipse((pmra[k], pmdec[k]), width=w, height=h,
                      angle=angle, fc='none', ec=fc, alpha=0.3, lw=0.5)
        ax.add_patch(ell)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel(r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)')
    ax.set_title(title)

axes[0].set_ylabel(r'$\mu_\delta$ (mas yr$^{-1}$)')
fig.suptitle(f'{FIELD_NAME} — VPD with 1σ error ellipses (N={N_ELLIPSE} brightest)',
             fontsize=12)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_vpd_ellipses.png', dpi=150)
plt.show()

In [ ]:
# ── Parallax vs Gaia G magnitude ──────────────────────────────────────────────
gmag_s = stars['gmag'].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vals, sig, title, color in [
    (axes[0],
     stars['parallax'].values,
     np.sqrt(C_gaia_stars[:, 4, 4]),
     'Gaia parallax', 'steelblue'),
    (axes[1],
     stars['parallax_bp3m'].values,
     stars['sigma_parallax_bp3m'].values,
     'BP3M parallax', 'darkorange'),
]:
    ok = np.isfinite(gmag_s) & np.isfinite(vals) & np.isfinite(sig)
    ax.errorbar(gmag_s[ok], vals[ok],
                yerr=sig[ok],
                fmt='none', ecolor=color, alpha=0.3, elinewidth=0.5)
    ax.scatter(gmag_s[ok], vals[ok], s=4, c=color, alpha=0.6, zorder=3)
    ax.axhline(0, color='k', lw=1, ls='--')
    ax.set_xlabel('Gaia G (mag)')
    ax.set_ylabel('Parallax (mas)')
    ax.set_title(title)

plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_parallax.png', dpi=150)
plt.show()

In [ ]:
# ── PM spatial maps — look for systematic residuals ───────────────────────────
ok = np.isfinite(pmra_b) & np.isfinite(pmdec_b)

ra  = stars['ra'].values[ok]
dec = stars['dec'].values[ok]
mu_a = pmra_b[ok]
mu_d = pmdec_b[ok]

# Flat-sky projection centred on the field (arcsec) for XY row
ra_cen  = np.median(ra)
dec_cen = np.median(dec)
x_arcsec = (ra  - ra_cen)  * np.cos(np.deg2rad(dec_cen)) * 3600.0
y_arcsec = (dec - dec_cen) * 3600.0

# Colour scale: symmetric ±3 MAD around median so outliers don't dominate
def sym_lim(arr, n=3):
    med = np.nanmedian(arr)
    mad = 1.4826 * np.nanmedian(np.abs(arr - med))
    return med - n * mad, med + n * mad

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for row_idx, (x, y, xlabel, ylabel) in enumerate([
    (ra,       dec,       'R.A. (deg)',          'Dec. (deg)'),
    (x_arcsec, y_arcsec,  r'$\Delta\alpha\cos\delta$ (arcsec)',
                           r'$\Delta\delta$ (arcsec)'),
]):
    for col_idx, (pm, label) in enumerate([
        (mu_a, r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)'),
        (mu_d, r'$\mu_\delta$ (mas yr$^{-1}$)'),
    ]):
        ax = axes[row_idx, col_idx]
        vmin, vmax = sym_lim(pm)
        sc = ax.scatter(x, y, c=pm, cmap='RdBu_r',
                        vmin=vmin, vmax=vmax,
                        s=6, alpha=0.8, rasterized=True)
        plt.colorbar(sc, ax=ax, label=label)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)

axes[0, 0].set_title(r'Sky map: $\mu_{\alpha^*}$')
axes[0, 1].set_title(r'Sky map: $\mu_\delta$')
axes[1, 0].set_title(r'Field position: $\mu_{\alpha^*}$')
axes[1, 1].set_title(r'Field position: $\mu_\delta$')

fig.suptitle(f'{FIELD_NAME} — Spatial PM maps (systematics check)', fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_pm_spatial.png', dpi=150)
plt.show()